# B_S2.1 — Observed Metric Distribution Visualization

Explores how metric values behave across function families and relationship
categories — **before** the permutation test.

| Section | Question |
|---------|----------|
| 1. Function family fingerprints | How does each family (+ Null) distribute on each metric? |
| 2. Null baseline deep dive | Can we separate True Null from Variance-only? |
| 3. Category overview | How do the 4 relationship categories separate? |
| 4. Core metrics vs SNR | How does signal strength affect metric values? |
| 5. Metric correlation | Which metrics carry redundant vs. complementary information? |
| 6. Spread pattern effect | How does heteroscedasticity shape metric behavior? |
| 7. X-distribution influence | Does sampling density bias metric values? |

In [1]:
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns
%matplotlib inline

S1_DIR = Path('output/S1')
S2_DIR = Path('output/S2')
VIZ_DIR = Path('output/S2/viz')
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Load case metadata
cases_main = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
cases_null = pd.read_csv(S1_DIR / 'null_expanded_cases.csv', low_memory=False)
cases_main['source'] = 'main'
cases_null['source'] = 'null_expanded'
offset = cases_main['case_id'].max()
cases_null['case_id'] = cases_null['case_id'] + offset
cases_df = pd.concat([cases_main, cases_null], ignore_index=True)

# Assign categories
is_null = cases_df['family_id'] == 'Null'
is_const = cases_df['spread_pattern'] == 'constant'
cases_df['category'] = 'mean+variance'
cases_df.loc[is_null & is_const, 'category'] = 'true_null'
cases_df.loc[~is_null & is_const, 'category'] = 'mean_only'
cases_df.loc[is_null & ~is_const, 'category'] = 'variance_only'

# Load metrics
metrics = pd.read_parquet(S2_DIR / 'metrics_full.parquet')
df = cases_df.merge(metrics, on='case_id', suffixes=('', '_m'))

print(f'Loaded {len(df):,} cases with {len(metrics.columns)} metric columns')
print(df['category'].value_counts())

Loaded 121,056 cases with 78 metric columns
category
mean+variance    85536
mean_only        28512
true_null         4032
variance_only     2976
Name: count, dtype: int64


In [ ]:
CAT_ORDER = ['true_null', 'variance_only', 'mean_only', 'mean+variance']
CAT_COLORS = {'true_null': '#999999', 'variance_only': '#f59e0b',
              'mean_only': '#3b82f6', 'mean+variance': '#10b981'}
CAT_LABELS = {'true_null': 'True Null', 'variance_only': 'Variance-only',
              'mean_only': 'Mean-only', 'mean+variance': 'Mean+Variance'}

CORE_METRICS = ['pearson_r', 'spearman_rho', 'distance_correlation', 'ew_bin_eta_squared']
CORE_LABELS = ['|Pearson r|', '|Spearman ρ|', 'Distance Correlation', 'η² (equal-width)']

# Use absolute values for correlation metrics
for col in ['pearson_r', 'spearman_rho']:
    if col in df.columns:
        df[col] = df[col].abs()

# Map eta_squared column name if needed
if 'equal_width_bin_eta_squared' in df.columns and 'ew_bin_eta_squared' not in df.columns:
    df['ew_bin_eta_squared'] = df['equal_width_bin_eta_squared']
if 'equal_count_bin_eta_squared' in df.columns and 'ec_bin_eta_squared' not in df.columns:
    df['ec_bin_eta_squared'] = df['equal_count_bin_eta_squared']

# --- Family short labels and TP-category color scheme ---
FAMILY_SHORT = {
    'F01': 'F01 Linear +',       'F02': 'F02 Linear −',
    'F03': 'F03 Power cvx +',    'F04': 'F04 Power cvx −',
    'F05': 'F05 Power ccv +',    'F06': 'F06 Power ccv −',
    'F07': 'F07 Saturation +',   'F08': 'F08 Saturation −',
    'F09': 'F09 Log +',          'F10': 'F10 Log −',
    'F11': 'F11 Exponential +',  'F12': 'F12 Exponential −',
    'F13': 'F13 S-curve +',      'F14': 'F14 S-curve −',
    'F15': 'F15 Threshold +',    'F16': 'F16 Threshold −',
    'F17': 'F17 Quad peak',      'F18': 'F18 Quad valley',
    'F19': 'F19 Spike',          'F20': 'F20 L-shaped',
    'F21': 'F21 Cubic',          'F22': 'F22 Complex',
    'Null (const σ)':  'Null (const σ)',
    'Null (varying σ)': 'Null (varying σ)',
}

# TP-based color mapping for family fingerprint plots
TP_MAP = {}
for i in range(1, 17):
    TP_MAP[f'F{i:02d}'] = '0 TP'
for i in range(17, 21):
    TP_MAP[f'F{i:02d}'] = '1 TP'
TP_MAP['F21'] = '2 TP'
TP_MAP['F22'] = 'Complex'
TP_MAP['Null (const σ)'] = 'Null'
TP_MAP['Null (varying σ)'] = 'Null (var)'

TP_COLORS = {
    '0 TP': '#3b82f6', '1 TP': '#f97316', '2 TP': '#22c55e',
    'Complex': '#a855f7', 'Null': '#9ca3af', 'Null (var)': '#f59e0b',
}

FAMILY_GROUP_ORDER = [f'F{i:02d}' for i in range(1, 23)] + ['Null (const σ)', 'Null (varying σ)']
FAMILY_GROUP_LABELS = [FAMILY_SHORT[g] for g in FAMILY_GROUP_ORDER]
FAMILY_GROUP_PALETTE = {FAMILY_SHORT[g]: TP_COLORS[TP_MAP[g]] for g in FAMILY_GROUP_ORDER}

## 1. Function Family Fingerprints

Horizontal boxplots showing how each of the 22 function families + 2 Null types
distribute on the 4 core metrics.

- **Signal families (F01–F22)**: mean-only cases (constant spread) across all SNR,
  to isolate the mean-response signal without spread-pattern confounding.
- **Null (const σ)**: True Null cases — f(X)=0, σ(X)=constant.
- **Null (varying σ)**: Variance-only cases — f(X)=0, σ(X) varies with x.

Color indicates turning-point category:
🔵 0 TP (monotonic) · 🟠 1 TP · 🟢 2 TP · 🟣 Complex · ⬜ Null (const) · 🟡 Null (varying)

In [ ]:
# --- Prepare profile data ---
# Signal families: mean-only (constant spread) to isolate mean response
signal_prof = df[df['category'] == 'mean_only'].copy()
signal_prof['group'] = signal_prof['family_id']

# Null: separate constant vs varying
null_c = df[df['category'] == 'true_null'].copy()
null_c['group'] = 'Null (const σ)'
null_v = df[df['category'] == 'variance_only'].copy()
null_v['group'] = 'Null (varying σ)'

profile_df = pd.concat([signal_prof, null_c, null_v], ignore_index=True)
profile_df['group_label'] = profile_df['group'].map(FAMILY_SHORT)

# --- Fig 1: Family fingerprint boxplots (horizontal, 24 rows × 4 metrics) ---
fig, axes = plt.subplots(1, 4, figsize=(24, 14), sharey=True)

for ax, metric, label in zip(axes, CORE_METRICS, CORE_LABELS):
    if metric not in profile_df.columns:
        ax.set_visible(False)
        continue
    sns.boxplot(data=profile_df, y='group_label', x=metric, ax=ax,
                order=FAMILY_GROUP_LABELS, hue='group_label',
                palette=FAMILY_GROUP_PALETTE,
                fliersize=0.3, linewidth=0.7, width=0.7, legend=False)
    # Null baseline reference line
    null_med = profile_df.loc[profile_df['group'] == 'Null (const σ)', metric].median()
    ax.axvline(null_med, color='#666666', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('')
    if ax == axes[0]:
        ax.tick_params(axis='y', labelsize=9)
        # Add horizontal separator lines between TP groups
        for sep_y in [15.5, 19.5, 20.5, 21.5]:
            ax.axhline(sep_y, color='grey', linewidth=0.4, linestyle=':', alpha=0.5)

plt.suptitle('Fig 1 — Function Family Metric Fingerprints\n'
             '(signal families: mean-only / constant spread / all SNR; '
             'dashed line = True Null baseline)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig1_family_fingerprints.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Fig 1b: Density overlay per metric — selected families ---
# Pick representatives: one per TP group + both null types
rep_groups = {
    'F01 Linear +': '#3b82f6',
    'F17 Quad peak': '#f97316',
    'F21 Cubic': '#22c55e',
    'F22 Complex': '#a855f7',
    'Null (const σ)': '#9ca3af',
    'Null (varying σ)': '#f59e0b',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, metric, label in zip(axes.flat, CORE_METRICS, CORE_LABELS):
    if metric not in profile_df.columns:
        ax.set_visible(False)
        continue
    for glabel, color in rep_groups.items():
        vals = profile_df.loc[profile_df['group_label'] == glabel, metric].dropna()
        if len(vals) > 0:
            ax.hist(vals, bins=50, alpha=0.4, color=color, density=True, label=glabel)
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.legend(fontsize=7, loc='upper right')
plt.suptitle('Fig 1b — Density Overlay: Representative Families\n'
             '(one per TP category + both Null types)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig1b_family_densities.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Median summary table ---
fam_medians = profile_df.groupby('group')[[m for m in CORE_METRICS if m in profile_df.columns]].median()
fam_medians = fam_medians.loc[FAMILY_GROUP_ORDER].round(4)
fam_medians.index = FAMILY_GROUP_LABELS
fam_medians.columns = CORE_LABELS[:len(fam_medians.columns)]
display(fam_medians)

## 2. Null Baseline Deep Dive

Zoomed comparison of the two Null types:
- **True Null** (f=0, constant σ): the "no relationship" ground truth
- **Variance-only** (f=0, varying σ): real dependence through variance

Key question: which metric separates them, and which cannot?

In [ ]:
null_df = df[df['family_id'] == 'Null'].copy()
null_df['null_type'] = null_df['category'].map({
    'true_null': 'True Null (const σ)',
    'variance_only': 'Variance-only (varying σ)',
})
null_type_order = ['True Null (const σ)', 'Variance-only (varying σ)']
null_palette = {null_type_order[0]: CAT_COLORS['true_null'],
                null_type_order[1]: CAT_COLORS['variance_only']}

# --- Fig 2a: Density overlay — True Null vs Variance-only ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, metric, label in zip(axes.flat, CORE_METRICS, CORE_LABELS):
    if metric not in null_df.columns:
        ax.set_visible(False)
        continue
    for nt, color in null_palette.items():
        vals = null_df.loc[null_df['null_type'] == nt, metric].dropna()
        ax.hist(vals, bins=60, alpha=0.5, color=color, density=True, label=nt)
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

    # Annotate separation
    med_tn = null_df.loc[null_df['null_type'] == null_type_order[0], metric].median()
    med_vo = null_df.loc[null_df['null_type'] == null_type_order[1], metric].median()
    sep = abs(med_vo - med_tn)
    ax.set_title(f'Δmedian = {sep:.4f}', fontsize=10, style='italic')

plt.suptitle('Fig 2a — Null Baseline: True Null vs Variance-only\n'
             '(large Δmedian = metric can separate them)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig2a_null_density.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Fig 2b: Horizontal boxplots — True Null vs Variance-only ---
fig, axes = plt.subplots(2, 2, figsize=(14, 5))
for ax, metric, label in zip(axes.flat, CORE_METRICS, CORE_LABELS):
    if metric not in null_df.columns:
        ax.set_visible(False)
        continue
    sns.boxplot(data=null_df, y='null_type', x=metric, ax=ax,
                order=null_type_order, hue='null_type', palette=null_palette,
                fliersize=1, linewidth=0.8, legend=False)
    ax.set_xlabel(label)
    ax.set_ylabel('')
plt.suptitle('Fig 2b — Null Baseline: Boxplot Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig2b_null_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Fig 2c: Variance-only breakdown by spread pattern ---
vo_df = null_df[null_df['category'] == 'variance_only'].copy()
spread_order = ['increasing', 'decreasing', 'middle_high']
spread_palette = {'increasing': '#10b981', 'decreasing': '#ef4444', 'middle_high': '#8b5cf6'}

fig, axes = plt.subplots(2, 2, figsize=(14, 5))
for ax, metric, label in zip(axes.flat, CORE_METRICS, CORE_LABELS):
    if metric not in vo_df.columns:
        ax.set_visible(False)
        continue
    sns.boxplot(data=vo_df, y='spread_pattern', x=metric, ax=ax,
                order=spread_order, hue='spread_pattern', palette=spread_palette,
                fliersize=1, linewidth=0.8, legend=False)
    ax.set_xlabel(label)
    ax.set_ylabel('')
plt.suptitle('Fig 2c — Variance-only: Breakdown by Spread Pattern',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig2c_variance_only_spread.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Summary: which metric separates Null types? ---
print('Median per null type:')
for m, l in zip(CORE_METRICS, CORE_LABELS):
    if m in null_df.columns:
        tn = null_df.loc[null_df['category'] == 'true_null', m].median()
        vo = null_df.loc[null_df['category'] == 'variance_only', m].median()
        ratio = vo / tn if tn > 1e-10 else float('inf')
        print(f'  {l:25s}  True Null={tn:.4f}  Var-only={vo:.4f}  ratio={ratio:.1f}×')

## 3. Category Overview

How do the 4 relationship categories separate on each metric?
Violin plots show both the distribution shape and density.

In [ ]:
# --- Fig 3: Category violin plots (horizontal) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, metric, label in zip(axes.flat, CORE_METRICS, CORE_LABELS):
    if metric not in df.columns:
        ax.set_visible(False)
        continue
    sns.violinplot(data=df, y='category', x=metric, ax=ax,
                   order=CAT_ORDER, hue='category', palette=CAT_COLORS,
                   inner='quartile', linewidth=0.8, cut=0, density_norm='width',
                   legend=False)
    ax.set_yticklabels([CAT_LABELS[c] for c in CAT_ORDER])
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('')
plt.suptitle('Fig 3 — Core Metric Distributions by Relationship Category (violin)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig3_category_violins.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Category summary table ---
available_core = [m for m in CORE_METRICS if m in df.columns]
cat_summary = df.groupby('category')[available_core].agg(['median', 'std']).round(4)
cat_summary = cat_summary.loc[CAT_ORDER]
display(cat_summary)

## 4. Core Metrics vs SNR

Mean-only cases (constant spread): how metric values rise with signal strength.

In [ ]:
mean_only_snr = df[df['category'] == 'mean_only'].copy()
mean_only_snr['snr_num'] = pd.to_numeric(mean_only_snr['snr'], errors='coerce')
mean_only_snr = mean_only_snr[mean_only_snr['snr_num'].notna() & np.isfinite(mean_only_snr['snr_num'])]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, metric, label in zip(axes.flat, CORE_METRICS, CORE_LABELS):
    if metric not in mean_only_snr.columns:
        ax.set_title(f'{label} (not available)')
        continue
    grouped = mean_only_snr.groupby('snr_num')[metric].agg(
        ['median', 'mean',
         lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]).reset_index()
    grouped.columns = ['snr', 'median', 'mean', 'q25', 'q75']
    ax.fill_between(grouped['snr'], grouped['q25'], grouped['q75'],
                    alpha=0.2, color='#3b82f6')
    ax.plot(grouped['snr'], grouped['median'], 'o-', color='#3b82f6',
            markersize=4, label='Median ± IQR')
    ax.set_xscale('log')
    ax.set_xlabel('SNR (log scale)')
    ax.set_ylabel(label)
    ax.set_title(f'{label} vs SNR', fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle('Fig 4 — Mean-only: Metric Response to Signal Strength',
             fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig4_snr_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Metric Correlation Matrix

Pairwise Pearson correlation of all numeric metrics. Identifies redundant vs. complementary metrics.

In [ ]:
metric_cols = [c for c in metrics.columns if c not in ('case_id', 'source', 'n_valid')
               and df[c].dtype in ('float64', 'float32', 'int64')]
metric_cols = [c for c in metric_cols if df[c].std() > 1e-10]

corr = df[metric_cols].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            xticklabels=True, yticklabels=True,
            cbar_kws={'shrink': 0.5, 'label': 'Pearson r'})
ax.set_title('Fig 5 — Metric Pairwise Correlation', fontsize=14)
ax.tick_params(labelsize=6)
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig5_metric_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Core 4 metrics: pairwise correlation
core_avail = [m for m in CORE_METRICS if m in df.columns]
core_corr = df[core_avail].corr().round(3)
core_corr.index = [l for m, l in zip(CORE_METRICS, CORE_LABELS) if m in df.columns]
core_corr.columns = core_corr.index
print('Core 4 metrics — pairwise correlation:')
display(core_corr)

## 6. Spread Pattern Effect

For a fixed function family (F01 Linear, SNR=2.0, even x), how do the 4 spread
patterns shift metric distributions?

In [ ]:
f01 = df[(df['family_id'] == 'F01') & (df['snr'] == 2.0) &
         (df['x_distribution'] == 'even')].copy()

spread_order = ['constant', 'increasing', 'decreasing', 'middle_high']
spread_colors = {'constant': '#3b82f6', 'increasing': '#10b981',
                 'decreasing': '#ef4444', 'middle_high': '#8b5cf6'}

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
for ax, metric, label in zip(axes.flat, CORE_METRICS, CORE_LABELS):
    if metric not in f01.columns:
        ax.set_visible(False)
        continue
    sns.boxplot(data=f01, y='spread_pattern', x=metric, ax=ax,
                order=spread_order, hue='spread_pattern', palette=spread_colors,
                fliersize=1, linewidth=0.8, legend=False)
    ax.set_xlabel(label)
    ax.set_ylabel('')
plt.suptitle('Fig 6 — F01 Linear (SNR=2.0, even): Spread Pattern Effect',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig6_spread_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. X-Distribution Influence

Does the sampling density of x bias metric values?
Mean-only cases at SNR=2.0, grouped by x_distribution.

In [ ]:
xdist_sub = df[(df['category'] == 'mean_only') & (df['snr'] == 2.0)].copy()
x_dist_order = ['even', 'left_dense', 'right_dense', 'center_dense',
                'clusters_2', 'clusters_3', 'clusters_4', 'clusters_5']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, metric, label in zip(axes.flat, CORE_METRICS, CORE_LABELS):
    if metric not in xdist_sub.columns:
        ax.set_visible(False)
        continue
    sns.boxplot(data=xdist_sub, y='x_distribution', x=metric, ax=ax,
                order=x_dist_order, color='#3b82f6',
                fliersize=0.5, linewidth=0.8)
    ax.set_xlabel(label)
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=9)
plt.suptitle('Fig 7 — X-Distribution Influence on Metrics (mean-only, SNR=2.0)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'fig7_xdist_influence.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary Statistics

In [ ]:
available_core = [m for m in CORE_METRICS if m in df.columns]
summary = df.groupby('category')[available_core].agg(['median', 'mean', 'std']).round(4)
summary = summary.loc[CAT_ORDER]
display(summary)

print()
print('=== Metric Statistics by Category ===')
for m, l in zip(CORE_METRICS, CORE_LABELS):
    if m in df.columns:
        for cat in CAT_ORDER:
            vals = df.loc[df['category'] == cat, m]
            print(f'  {l:25s}  {CAT_LABELS[cat]:15s}  '
              f'median={vals.median():.4f}  mean={vals.mean():.4f}  std={vals.std():.4f}')